<a href="https://colab.research.google.com/github/suhani8338/word_analysis_of_the_edict/blob/main/WordAnalysisOfTheEdict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas lxml nltk scikit-learn tqdm

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import csv
import os
import re
import time
import json
from collections import deque
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

# ---------------- Configuration ----------------

# Hubs on the main site
SECTION_URLS = [
    "https://www.the-edict.in/sg-newsdesk",
    "https://www.the-edict.in/opinions",
]

# WordPress-style archive tags to paginate completely (legacy archive)
TAG_ARCHIVES = [
    "http://edictarchive.the-edict.in/index.php/tag/politics/",
    "http://edictarchive.the-edict.in/index.php/tag/opinion/",
]

OUT_DIR = "corpus"
CSV_PATH = os.path.join(OUT_DIR, "corpus.csv")
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/119.0.0.0 Safari/537.36"
)
ACCEPT_LANGUAGE = "en-US,en;q=0.9"
SLEEP_SECS = 1.0
TIMEOUT = 20
MAX_PAGES = 2000  # safety cap for archive pagination

os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- HTTP session ----------------

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": ACCEPT_LANGUAGE,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Encoding": "identity",
})

def get(url, referer=None):
    headers = {}
    if referer:
        headers["Referer"] = referer
    resp = session.get(url, timeout=TIMEOUT, headers=headers, allow_redirects=True)
    # Improve decoding for older WP pages
    if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding or "utf-8"
    # Retry once if tiny or soft-blocked, with a site-root referer
    if (len(resp.text) < 1000 or resp.status_code in (403, 503)) and not referer:
        parts = urlparse(url)
        root = f"{parts.scheme}://{parts.netloc}/"
        resp = session.get(url, timeout=TIMEOUT, headers={"Referer": root}, allow_redirects=True)
        if not resp.encoding or resp.encoding.lower() == "iso-8859-1":
            resp.encoding = resp.apparent_encoding or "utf-8"
    resp.raise_for_status()
    return resp

def soup_from_url(url):
    html = get(url).text
    return BeautifulSoup(html, "lxml")

# ---------------- URL utilities ----------------

def normalize(u):
    parts = urlparse(u)
    clean = parts._replace(fragment="", query="").geturl()
    if clean.endswith("/"):
        clean = clean[:-1]
    return clean

def same_host(u, root):
    return urlparse(u).netloc == urlparse(root).netloc

def looks_like_article_url(u):
    path = urlparse(u).path
    if path in ["/", "", "/sg-newsdesk", "/opinions"]:
        return False
    segments = [s for s in path.split("/") if s]
    if "/post/" in path:
        return True
    # Accept WP date paths, including /index.php/YYYY/MM/
    if re.search(r"/\d{4}/\d{2}/", path):
        return True
    # Broad heuristic for content pages
    return len(segments) >= 2 and not any(seg in {"category", "categories", "tag", "tags", "author"} for seg in segments)

# ---------------- Hub crawling ----------------

def find_article_links(section_url):
    links = set()
    try:
        s = soup_from_url(section_url)
    except Exception:
        return links
    for a in s.select("a[href]"):
        href = a.get("href")
        if not href:
            continue
        absu = normalize(urljoin(section_url, href))
        if not absu.startswith("http"):
            continue
        if not same_host(absu, section_url):
            continue
        if looks_like_article_url(absu):
            links.add(absu)
    return links

# -------- WordPress tag archive crawling with pagination --------

TAG_ARTICLE_SELECTORS = [
    "h1.entry-title a",
    "h2.entry-title a",
    "h3.entry-title a",
    "h4.entry-title a",
    "article header h1 a",
    "article header h2 a",
    ".entry-title a",
    "a.more-link",          # Read more / Keep Reading
    "a[rel='bookmark']",
    ".post-title a",
]

def find_read_more_links(soup, base_url):
    found = set()
    for a in soup.select("a[href]"):
        txt = (a.get_text() or "").strip().lower()
        if "keep reading" in txt or "read more" in txt:
            absu = normalize(urljoin(base_url, a["href"]))
            found.add(absu)
    return found

def find_tag_article_links_on_page(tag_page_url, soup):
    links = set()
    for sel in TAG_ARTICLE_SELECTORS:
        for a in soup.select(sel):
            href = a.get("href")
            if not href:
                continue
            absu = normalize(urljoin(tag_page_url, href))
            if looks_like_article_url(absu):
                links.add(absu)
    links |= find_read_more_links(soup, tag_page_url)
    return links

PAGINATION_TEXT_HINTS = [
    "older", "previous", "older posts", "older entries",
    "next", "newer", "newer posts", "newer entries",
]

def find_pagination_links(tag_page_url, soup):
    links = set()
    # rel-based
    for rel in ["next", "prev", "previous"]:
        for a in soup.select(f"a[rel='{rel}']"):
            href = a.get("href")
            if not href:
                continue
            absu = normalize(urljoin(tag_page_url, href))
            links.add(absu)
    # text-based
    for a in soup.select("a[href]"):
        txt = (a.get_text() or "").strip().lower()
        if any(h in txt for h in PAGINATION_TEXT_HINTS):
            absu = normalize(urljoin(tag_page_url, a["href"]))
            links.add(absu)
    # pattern-based /page/N/
    base_path = urlparse(tag_page_url).path.rstrip("/") + "/"
    for a in soup.select("a[href]"):
        href = a.get("href")
        if not href:
            continue
        absu = normalize(urljoin(tag_page_url, href))
        p = urlparse(absu).path
        if base_path in p and re.search(r"/page/\d+/?$", p):
            links.add(absu)
    return links

def crawl_wordpress_tag_archive(start_url, max_pages=MAX_PAGES):
    visited_pages = set()
    to_visit = deque([start_url])
    article_links = set()
    pages_seen = 0

    while to_visit and pages_seen < max_pages:
        url = normalize(to_visit.popleft())
        if url in visited_pages:
            continue
        try:
            soup = soup_from_url(url)
        except Exception:
            continue
        visited_pages.add(url)
        pages_seen += 1

        article_links |= find_tag_article_links_on_page(url, soup)
        for nxt in find_pagination_links(url, soup):
            if nxt not in visited_pages:
                to_visit.append(nxt)

        time.sleep(SLEEP_SECS)

    return article_links

# ------------------- Article extraction -------------------

def extract_jsonld_article(soup):
    data = {}
    for tag in soup.select('script[type="application/ld+json"]'):
        raw = tag.string or tag.get_text() or ""
        raw = raw.strip()
        if not raw:
            continue
        try:
            payload = json.loads(raw)
        except Exception:
            continue
        items = payload if isinstance(payload, list) else [payload]
        for it in items:
            typ = it.get("@type", "")
            types = [typ] if isinstance(typ, str) else typ if isinstance(typ, list) else []
            types = [t.lower() for t in types if isinstance(t, str)]
            if any(t in {"article", "newsarticle", "blogposting"} for t in types):
                data["title"] = it.get("headline") or it.get("name") or data.get("title")
                author = it.get("author")
                if isinstance(author, dict):
                    data["author"] = author.get("name")
                elif isinstance(author, list) and author:
                    first = author[0]
                    if isinstance(first, dict):
                        data["author"] = first.get("name")
                data["date"] = it.get("datePublished") or data.get("date")
                return data
    return data

def meta_content(soup, names):
    for n in names:
        tag = soup.find("meta", attrs={"property": n}) or soup.find("meta", attrs={"name": n})
        if tag and tag.get("content"):
            return tag["content"].strip()
    return None

def extract_title(soup):
    return (
        meta_content(soup, ["og:title", "twitter:title"])
        or (soup.find("h1").get_text(strip=True) if soup.find("h1") else None)
        or (soup.title.get_text(strip=True) if soup.title else None)
        or ""
    )

def extract_author(soup):
    # WordPress often uses meta[name='author']; external meta may be missing
    author = meta_content(soup, ["article:author", "author"])
    if author:
        return author
    # Try byline patterns
    by = soup.select_one(".byline, .post-meta, .posted-on, .author, .entry-meta")
    if by:
        txt = by.get_text(" ", strip=True)
        if txt:
            return txt
    return ""

def extract_date(soup):
    date = meta_content(soup, ["article:published_time", "date", "datePublished"])
    if date:
        return date
    time_tag = soup.select_one("time[datetime]")
    if time_tag and time_tag.get("datetime"):
        return time_tag["datetime"].strip()
    return ""

CONTENT_SELECTORS = [
    # WordPress-first
    "div.entry-content",
    "article .entry-content",
    "div.post-content",
    "article .post-content",
    "div.td-post-content",

    # Common legacy WP themes
    ".single-content",
    ".post-inner",
    ".post-entry",
    ".post .the-content",
    ".hentry .entry",
    ".hentry",
    "#content .post",
    "#content .entry",
    "#primary .site-main .post",
    ".site-content .post",
    ".single-post .post",
    ".single .post",
    ".post",

    # Generic
    "article",
    "main",
    "section[role='main']",
    "div[data-hook='post-content']",
    "div[data-hook='post-body']",
    "div[id*='content']",
    "div[class*='content']",
]

def clean_text(txt):
    txt = re.sub(r"\r\n?", "\n", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

def extract_body(soup):
    # Drop obvious boilerplate/noise before extracting
    for junk in soup.select(
        "header, footer, nav, aside, form, script, style, noscript, "
        ".sharedaddy, .jp-relatedposts, .post-navigation, .comments, "
        ".site-footer, .sidebar, .widget, .related, .advert, .ads"
    ):
        junk.decompose()

    # 1) Structured extraction: paragraphs/headings/lists/quotes from known containers
    def pull_from(node):
        parts = [
            t.get_text(" ", strip=True)
            for t in node.select("p, h2, h3, li, blockquote")
            if t.get_text(strip=True)
        ]
        if parts:
            return clean_text("\n\n".join(parts))
        # Many legacy posts have <div> + <br>; fall back to stripped strings
        raw = " ".join(s for s in node.stripped_strings)
        return clean_text(raw)

    for sel in CONTENT_SELECTORS:
        node = soup.select_one(sel)
        if node:
            text = pull_from(node)
            if len(text.split()) >= 50:
                return text

    # 2) Container sweep: pick the single largest text block from likely containers
    candidates = soup.select(
        "article, #content, .site-content, #primary, .content-area, "
        ".hentry, .single, .single-post, .post, .entry"
    )
    best_text, best_wc = "", 0
    for c in candidates:
        txt = pull_from(c)
        wc = len(txt.split())
        if wc > best_wc:
            best_text, best_wc = txt, wc
    if best_wc >= 50:
        return best_text

    # 3) Final fallback: document-level largest block (still cleaned)
    doc_txt = clean_text(" ".join(soup.stripped_strings))
    return doc_txt

def is_article_page(soup):
    # Fast positives
    ogtype = meta_content(soup, ["og:type"])
    if ogtype and "article" in ogtype.lower():
        return True

    j = extract_jsonld_article(soup)
    if j.get("title") or j.get("date") or j.get("author"):
        return True

    # WP hints and single-post body classes
    if soup.select_one("h1.entry-title") or soup.select_one("div.entry-content"):
        return True
    body = soup.find("body")
    if body and any(c in body.get("class", []) for c in ["single", "single-post", "single-format-standard"]):
        return True

    # Fallback by body length
    body_text = extract_body(soup)
    return len(body_text.split()) >= 100

def slugify(url):
    path = urlparse(url).path.strip("/")
    if not path:
        return "index"
    slug = re.sub(r"[^a-zA-Z0-9\\-_/]", "-", path)
    slug = slug.replace("/", "_")
    return slug[:150]

def scrape_article(url):
    s = soup_from_url(url)
    if not is_article_page(s):
        return None
    body = extract_body(s)
    meta = extract_jsonld_article(s)
    title = meta.get("title") or extract_title(s)
    author = meta.get("author") or extract_author(s)
    date = meta.get("date") or extract_date(s)
    return {
        "url": url,
        "title": title,
        "author": author,
        "date": date,
        "text": body,
        "word_count": len(body.split()),
    }

def write_csv(rows, path):
    fieldnames = ["url", "title", "author", "date", "word_count", "text"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fieldnames})

def write_txt(row):
    slug = slugify(row["url"])
    fp = os.path.join(OUT_DIR, f"{slug}.txt")
    with open(fp, "w", encoding="utf-8") as f:
        f.write(f"{row['title']}\n")
        f.write(f"Author: {row['author']}\n")
        f.write(f"Date: {row['date']}\n")
        f.write(f"URL: {row['url']}\n")
        f.write("\n")
        f.write(row["text"])

def main():
    all_links = set()

    # Crawl main hubs
    for hub in SECTION_URLS:
        try:
            links = find_article_links(hub)
            print(f"[hub] {hub} -> {len(links)} candidate links")
            all_links.update(links)
        except Exception as e:
            print(f"Failed to parse hub {hub}: {e}")
        time.sleep(SLEEP_SECS)

    # Crawl paginated WordPress archives
    for tag_url in TAG_ARCHIVES:
        try:
            tag_links = crawl_wordpress_tag_archive(tag_url)
            print(f"[tag] {tag_url} -> {len(tag_links)} article links")
            all_links.update(tag_links)
        except Exception as e:
            print(f"Failed to crawl tag {tag_url}: {e}")
        time.sleep(SLEEP_SECS)

    print(f"Total unique candidates: {len(all_links)}")

    rows = []
    for i, url in enumerate(sorted(all_links)):
        try:
            art = scrape_article(url)
            if art and art["text"]:
                rows.append(art)
                write_txt(art)
                print(f"[{i+1}/{len(all_links)}] OK: {url} ({art['word_count']} words)")
            else:
                print(f"[{i+1}/{len(all_links)}] Skip (not article or empty): {url}")
        except Exception as e:
            print(f"[{i+1}/{len(all_links)}] Error {url}: {e}")
        time.sleep(SLEEP_SECS)

    if rows:
        write_csv(rows, CSV_PATH)
        print(f"Wrote {len(rows)} articles to {CSV_PATH}")
    else:
        print("No articles extracted")

if __name__ == "__main__":
    main()


[hub] https://www.the-edict.in/sg-newsdesk -> 22 candidate links
[hub] https://www.the-edict.in/opinions -> 28 candidate links
[tag] http://edictarchive.the-edict.in/index.php/tag/politics/ -> 45 article links
[tag] http://edictarchive.the-edict.in/index.php/tag/opinion/ -> 46 article links
Total unique candidates: 133
[1/133] OK: http://edictarchive.the-edict.in/index.php/2016/04/28/they-are-watching-us (254 words)
[2/133] OK: http://edictarchive.the-edict.in/index.php/2016/10/15/breaking-down-the-kashmir-petition-what-you-need-to-know (1961 words)
[3/133] OK: http://edictarchive.the-edict.in/index.php/2017/11/07/all-the-things-i-wish-i-said-at-the-sexual-harassment-in-academia-open-house-but-i-didnt (1775 words)
[4/133] OK: http://edictarchive.the-edict.in/index.php/2017/11/07/sexual-harassment-on-campus (877 words)
[5/133] OK: http://edictarchive.the-edict.in/index.php/2017/11/15/engendering-discussion-in-conversation-with-manjari-sahay (2615 words)
[6/133] OK: http://edictarchive.t

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
import sys
from collections import Counter
from typing import List, Tuple

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import nltk
from nltk.corpus import stopwords

# =========================
# CONFIGURABLE PARAMETERS
# =========================
csv_path = "corpus (2).csv"  # ✅ <-- set this to your CSV file path
text_column = "text"                # column name containing text data

model_name = "Qwen/Qwen3-Embedding-0.6B"
seeds = ["struggle", "campus", "nationalism", "radical", "freedom", "discrimination", "workers", "dalit", "feminism", "caste"]
topk = 20
vocab_max = 10000
min_freq = 2
extra_exclude = seeds  # can add more words here if needed
# =========================


def ensure_nltk():
    try:
        _ = stopwords.words("english")
    except LookupError:
        nltk.download("stopwords", quiet=True)


def tokenize(text: str) -> List[str]:
    tokens = re.findall(r"[A-Za-z][A-Za-z\-']+", text)
    return [t.lower() for t in tokens]


def build_vocab(tokens: List[str], extra_exclude: List[str], max_vocab: int) -> Tuple[List[str], Counter]:
    sw = set(stopwords.words("english"))
    sw.update([w.lower() for w in extra_exclude])
    clean = [t for t in tokens if t not in sw and len(t) >= 3 and re.fullmatch(r"[a-z][a-z\-']+", t)]
    freqs = Counter(clean)
    vocab = [w for w, _ in freqs.most_common(max_vocab)]
    return vocab, freqs


def embed_words(model: SentenceTransformer, words: List[str], batch_size: int = 64) -> np.ndarray:
    return model.encode(
        words,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


def top_k_similar(seed: str, seed_vec: np.ndarray, vocab: List[str],
                  vocab_vecs: np.ndarray, k: int, exclude: set):
    sims = cosine_similarity(seed_vec.reshape(1, -1), vocab_vecs).ravel()
    ranked = [(vocab[i], float(sims[i])) for i in np.argsort(-sims)]
    out = []
    for w, s in ranked:
        if w in exclude:
            continue
        out.append((w, s))
        if len(out) >= k:
            break
    return out


# ========== MAIN EXECUTION ==========
ensure_nltk()

# Read all text from CSV
try:
    df = pd.read_csv(csv_path)
    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in CSV.")
    all_text = " ".join(str(t) for t in df[text_column].dropna())
except Exception as e:
    print(f"❌ Error reading CSV: {e}")
    sys.exit(1)

if not all_text.strip():
    print("❌ No text found in the CSV file.")
else:
    tokens = tokenize(all_text)
    vocab, freqs = build_vocab(tokens, extra_exclude=extra_exclude, max_vocab=vocab_max)
    vocab = [w for w in vocab if freqs[w] >= min_freq]

    print("\n=== Top 25 non-stopword terms ===")
    for w, c in freqs.most_common(25):
        print(f"{w}\t{c}")

    if not vocab:
        print("\nNo vocabulary after filtering; try lowering min_freq or increasing vocab_max.")
    else:
        model = SentenceTransformer(model_name)
        vocab_vecs = embed_words(model, vocab)
        seed_vecs = embed_words(model, [s.lower() for s in seeds])

        print("\n=== Per-seed clusters (top-k nearest words) ===")
        for si, seed in enumerate(seeds):
            neighbors = top_k_similar(
                seed=seed,
                seed_vec=seed_vecs[si],
                vocab=vocab,
                vocab_vecs=vocab_vecs,
                k=topk,
                exclude=set(seeds),
            )
            print(f"\nSeed: {seed}")
            for rank, (w, sim) in enumerate(neighbors, 1):
                freq = freqs.get(w, 0)
                print(f"{rank:2d}. {w}\t(sim={sim:.4f}, freq={freq})")



=== Top 25 non-stopword terms ===
students	623
student	622
ashoka	554
university	367
one	352
also	326
house	280
party	258
would	238
body	237
like	213
first	211
election	211
people	199
administration	178
reply	178
members	175
auec	164
time	163
new	161
issues	157
candidates	155
even	148
system	147
politics	145


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== Per-seed clusters (top-k nearest words) ===

Seed: struggle
 1. struggles	(sim=0.8879, freq=3)
 2. fighting	(sim=0.8571, freq=5)
 3. conflict	(sim=0.8564, freq=23)
 4. struggling	(sim=0.8540, freq=3)
 5. fought	(sim=0.8359, freq=6)
 6. fight	(sim=0.8244, freq=11)
 7. confront	(sim=0.8150, freq=3)
 8. overcome	(sim=0.8138, freq=6)
 9. contesting	(sim=0.8121, freq=19)
10. contested	(sim=0.8104, freq=9)
11. strive	(sim=0.8095, freq=4)
12. defense	(sim=0.7919, freq=2)
13. battle	(sim=0.7914, freq=2)
14. strategy	(sim=0.7908, freq=14)
15. strength	(sim=0.7907, freq=9)
16. strike	(sim=0.7885, freq=12)
17. war	(sim=0.7818, freq=10)
18. protest	(sim=0.7778, freq=64)
19. attack	(sim=0.7769, freq=2)
20. opposition	(sim=0.7767, freq=12)

Seed: campus
 1. campuses	(sim=0.9519, freq=3)
 2. on-campus	(sim=0.8997, freq=2)
 3. campus-wide	(sim=0.8799, freq=2)
 4. university	(sim=0.8368, freq=367)
 5. school	(sim=0.8236, freq=13)
 6. institutions	(sim=0.8225, freq=23)
 7. institute	(sim=0.8192, fr

In [ ]:
from collections import Counter

# We will re-calculate counts from the original 'tokens' list.
# The 'freqs' variable from cell 1 cannot be used, as it
# was created *after* removing all stopwords and seed words.

# 'tokens' and 'seeds' are variables already in memory from cell 1.
print("Recalculating frequencies for all words...")
all_token_freqs = Counter(tokens)

print("\n=== Seed Word Frequencies ===")
print("---------------------------------")

# Loop through your seed list and find the count for each
for seed in seeds:
    # Use .get(seed, 0) to safely get the count (it returns 0 if not found)
    frequency = all_token_freqs.get(seed, 0)

    # Simple formatting to align the columns
    if len(seed) < 8:
        print(f"{seed}\t\t{frequency}")
    else:
        print(f"{seed}\t{frequency}")

print("---------------------------------")

Recalculating frequencies for all words...

=== Seed Word Frequencies ===
---------------------------------
struggle	10
campus		289
nationalism	3
radical		8
freedom		36
discrimination	11
workers		181
dalit		0
feminism	1
caste		23
---------------------------------


In [ ]:
import os
from wordcloud import WordCloud


output_dir = "word_clouds"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving word clouds to: {output_dir}/")

print("\n=== Generating Word Clouds ===")

for si, seed in enumerate(seeds):
    neighbors = top_k_similar(
        seed=seed,
        seed_vec=seed_vecs[si],
        vocab=vocab,
        vocab_vecs=vocab_vecs,
        k=topk,
        exclude=set(seeds),
    )

    print(f"\nProcessing seed: {seed}")

    if neighbors:
        sim_scores = dict(neighbors)

        # Initialize the word cloud generator
        wc = WordCloud(
            width=1000,
            height=500,
            background_color="white",
            colormap="viridis"  # You can change the color map if you like
        )

        # Generate the cloud from the similarity scores
        wc.generate_from_frequencies(sim_scores)

        # Define the output filename and path
        cloud_filename = f"wordcloud_{seed}.png"
        cloud_filepath = os.path.join(output_dir, cloud_filename)

        try:
            # Save the generated image to a file
            wc.to_file(cloud_filepath)
            print(f"✅ Generated word cloud: {cloud_filepath}")
        except Exception as e:
            print(f"❌ Error saving word cloud for '{seed}': {e}")
    else:
        print(f"No neighbors found for '{seed}', skipping word cloud.")

Saving word clouds to: word_clouds/

=== Generating Word Clouds ===

Processing seed: struggle
✅ Generated word cloud: word_clouds/wordcloud_struggle.png

Processing seed: campus
✅ Generated word cloud: word_clouds/wordcloud_campus.png

Processing seed: nationalism
✅ Generated word cloud: word_clouds/wordcloud_nationalism.png

Processing seed: radical
✅ Generated word cloud: word_clouds/wordcloud_radical.png

Processing seed: freedom
✅ Generated word cloud: word_clouds/wordcloud_freedom.png

Processing seed: discrimination
✅ Generated word cloud: word_clouds/wordcloud_discrimination.png

Processing seed: workers
✅ Generated word cloud: word_clouds/wordcloud_workers.png

Processing seed: dalit
✅ Generated word cloud: word_clouds/wordcloud_dalit.png

Processing seed: feminism
✅ Generated word cloud: word_clouds/wordcloud_feminism.png

Processing seed: caste
✅ Generated word cloud: word_clouds/wordcloud_caste.png
